In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-32B-AWQ",
    tensor_parallel_size=1,
    quantization="awq",
    max_model_len=8192,
    trust_remote_code=True,
)

print("✓ Model loaded successfully!")

WARNING 07-22 15:25:01 [config.py:71] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
INFO 07-22 15:25:03 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen3-32B-AWQ'}
INFO 07-22 15:25:03 [model.py:611] Resolved architecture: Qwen3ForCausalLM
INFO 07-22 15:25:03 [model.py:1745] Using max model len 8192
INFO 07-22 15:25:04 [awq_marlin.py:273] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 07-22 15:25:04 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-22 15:25:05 [vllm.py:999] Asynchronous scheduling is enabled.
INFO 07-22 15:25:05 [kernel.py:270] Final 

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=561) INFO 07-22 15:25:19 [default_loader.py:397] Loading weights took 6.82 seconds
(EngineCore pid=561) INFO 07-22 15:25:20 [gpu_model_runner.py:5187] Model loading took 18.24 GiB memory and 9.079509 seconds
(EngineCore pid=561) INFO 07-22 15:25:28 [backends.py:1089] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/e2f8214a4b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=561) INFO 07-22 15:25:28 [backends.py:1148] Dynamo bytecode transform time: 7.40 s
(EngineCore pid=561) INFO 07-22 15:25:32 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 3.732 s
(EngineCore pid=561) INFO 07-22 15:25:32 [decorators.py:311] Directly load AOT compilation from path /home/jovyan/.cache/vllm/torch_compile_cache/torch_aot_compile/d63130785d34e0c32e4910307d191df28e2ee6230c522344427983b01aaaed77/rank_0_0/model
(EngineCore pid=561) INFO 07-22 15:25:32 [monitor.py:53] torch.compile took 12.08 s in total
(Engin

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:22<00:00,  2.26it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:13<00:00,  2.53it/s]


(EngineCore pid=561) INFO 07-22 15:26:20 [gpu_model_runner.py:6585] Graph capturing finished in 38 secs, took 1.54 GiB
(EngineCore pid=561) INFO 07-22 15:26:20 [gpu_worker.py:639] CUDA graph pool memory: 1.54 GiB (actual), 1.49 GiB (estimated), difference: 0.05 GiB (3.0%).
(EngineCore pid=561) INFO 07-22 15:26:20 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=561) INFO 07-22 15:26:20 [core.py:306] init engine (profile, create kv cache, warmup model) took 60.66 s (compilation: 12.08 s)
(EngineCore pid=561) INFO 07-22 15:26:20 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
✓ Model loaded successfully!
(EngineCore pid=561) WARNING 07-22 15:27:36 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover thi

## Prepare a random sample from the training dataframe

In [2]:
import pandas as pd
import json
import re
from vllm import SamplingParams

full_df = pd.read_csv('../data/full_df_with_partitions.csv', index_col=0)
train_df = full_df[full_df['split']=='train']

non_label_codes = ['Correct', 'Neither']

train_df = train_df.copy()
train_df['label_is_correct'] = train_df['FinalCode'] == 'Correct'
train_df['label_is_misunderstanding'] = ~train_df['FinalCode'].isin(non_label_codes)
train_df['label_which_misunderstanding'] = train_df['FinalCode'].where(~train_df['FinalCode'].isin(non_label_codes), other='')

# Get a sample
sample_df = train_df.sample(n=1000, random_state=42).reset_index(drop=True)

sample_df.head()

,QuestionId,Question.Text,MC_Answer,StudentExplanation,FinalCode,Correct,split,label_is_correct,label_is_misunderstanding,label_which_misunderstanding
0,33472,\( \frac{1}{3}+\frac{2}{5}= \),\( \frac{3}{15} \),3 and 5 go into 15 so the domminater is 15 and...,Denominator-only change,False,train,False,True,Denominator-only change
1,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),the dots are going up in 4,Correct,True,train,True,False,
2,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),because it goes up by five and then u add two ...,Neither,True,train,False,False,
3,31777,A box contains \( 120 \) counters. The counter...,\( 72 \),I think this because 1/5 of 120 is 24 and 3/5 ...,Correct,True,train,True,False,
4,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 3 \),i was just guessing because i haven't gone ove...,Neither,False,train,False,False,


## Prepare input frame and output schema

In [11]:
import pandas as pd
import json
import re
from pydantic import BaseModel
from vllm import SamplingParams

# Define the exact output schema
class MisconceptionLabel(BaseModel):
    is_correct: bool
    is_misunderstanding: bool
    which_misunderstanding: str

json_schema = MisconceptionLabel.model_json_schema()

MISCONCEPTION_DEFINITION = """
A "misunderstanding" is a systematic error pattern, NOT a simple calculation slip, guess, or vague/unclear response. There are two types. 
Conceptual misunderstandings stem from a wrong mental model of a math concept. Examples: whole-number bias (thinking 3/9 > 2/5 because 3>2 and 9>5; thinking 0.15 > 0.4 because 15>4; thinking zeros after the decimal point don't matter; thinking multiplication always produces a larger number), thinking "=" means "compute an answer" rather than representing equivalence, thinking a variable just means "a missing number" rather than a quantity that can be solved for.
Procedural misunderstandings come from applying the wrong procedure consistently, often by misapplying a rule from one operation to another. Examples: adding fraction numerators and denominators independently like whole numbers (3/5 + 3/5 = 6/10), using a common denominator when multiplying fractions instead of multiplying the denominator (3/5 * 2/5 = 6/5), or inverting the wrong fraction during division.

A misunderstanding must reflect a CONSISTENT, SYSTEMATIC pattern of incorrect reasoning. 
A simple arithmetic slip (e.g., 2 x 8 = 18), a guess, or a vague/unexplained answer is NOT a misunderstanding.
"""

def build_prompt(row):
    return f"""You are analyzing a student's response to a math question to determine 
whether their self-explanation is correct, and if incorrect, whether it reflects a genuine 
mathematical misunderstanding (as opposed to a simple slip, guess, or vague response).

DEFINITION OF MISUNDERSTANDING:
{MISCONCEPTION_DEFINITION}

QUESTION: {row['Question.Text']}
STUDENT'S SELF-EXPLANATION: {row['StudentExplanation']}
STUDENT'S MULTIPLE CHOICE ANSWER: {row['MC_Answer']}

Based on the above, determine:
1. is_correct: Is the student's self-explanation mathematically correct?
2. is_misunderstanding: If incorrect, does the explanation reveal a genuine systematic 
   misunderstanding (per the definition above), as opposed to a calculation slip, guess, 
   or vague/unexplained response?
3. which_misunderstanding: If is_misunderstanding is true, briefly describe the student's thought process that led to the misunderstanding with a 1-3 word label. 
   If is_misunderstanding is false, leave this as an empty string.
"""

# Make a list of conversations
conversations = [
    [{"role": "user", "content": build_prompt(row)}]
    for _, row in sample_df.iterrows()
]

# Define structured output
from vllm.sampling_params import StructuredOutputsParams
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    structured_outputs=StructuredOutputsParams(json=json_schema),
)

# Run model and get results

In [12]:
# Run inference, disabling Qwen3's "thinking" mode so output is just the JSON
outputs = llm.chat(
    conversations,
    sampling_params,
    chat_template_kwargs={"enable_thinking": False},
)

# Parsing should now basically always succeed, but keep a safety net
def parse_response(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(text)
        return {
            "is_correct": parsed.get("is_correct"),
            "is_misunderstanding": parsed.get("is_misunderstanding"),
            "which_misunderstanding": parsed.get("which_misunderstanding", ""),
        }
    except json.JSONDecodeError:
        return {"is_correct": None, "is_misunderstanding": None, "which_misunderstanding": None}

parsed_results = [parse_response(o.outputs[0].text) for o in outputs]

results_df = sample_df.copy()
results_df["is_correct"] = [r["is_correct"] for r in parsed_results]
results_df["is_misunderstanding"] = [r["is_misunderstanding"] for r in parsed_results]
results_df["which_misunderstanding"] = [r["which_misunderstanding"] for r in parsed_results]

# Enforce business rules
results_df.loc[results_df["is_correct"] == True, "is_misunderstanding"] = False
results_df.loc[results_df["is_misunderstanding"] == False, "which_misunderstanding"] = ""

# Sanity check: should now be 0 or near-0
print(f"Remaining parse failures: {results_df['is_correct'].isna().sum()}")
results_df.head(10)

Rendering conversations:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1000/1000 [03:00<00:00,  5.55it/s, est. speed input: 3127.71 toks/s, output: 176.75 toks/s]

Remaining parse failures: 0


,QuestionId,Question.Text,MC_Answer,StudentExplanation,FinalCode,Correct,split,label_is_correct,label_is_misunderstanding,label_which_misunderstanding,is_correct,is_misunderstanding,which_misunderstanding
0,33472,\( \frac{1}{3}+\frac{2}{5}= \),\( \frac{3}{15} \),3 and 5 go into 15 so the domminater is 15 and...,Denominator-only change,False,train,False,True,Denominator-only change,False,True,Adding numerators and denominators independently
1,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),the dots are going up in 4,Correct,True,train,True,False,,True,False,
2,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),because it goes up by five and then u add two ...,Neither,True,train,False,False,,False,True,inconsistent pattern reasoning
3,31777,A box contains \( 120 \) counters. The counter...,\( 72 \),I think this because 1/5 of 120 is 24 and 3/5 ...,Correct,True,train,True,False,,True,False,
4,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 3 \),i was just guessing because i haven't gone ove...,Neither,False,train,False,False,,False,False,
5,91695,Dots have been arranged in these patterns: [Im...,\( 22 \),So we addd four in of them 18+4 in total.,Wrong term,False,train,False,True,Wrong term,True,False,
6,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),becausee there are 9 total and three are not s...,Incomplete,False,train,False,True,Incomplete,False,True,whole-number bias
7,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),you do what do you do to the bottom to the top...,Additive,True,train,False,True,Additive,False,True,procedural
8,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),6/10 is equivalent to 9/15 because 5x2=10 and ...,Correct,True,train,True,False,,False,True,whole-number bias
9,32833,Calculate \( \frac{2}{3} \times 5 \),\( 3 \frac{1}{3} \),i think this is because i did 2 1/3 times 5/1 ...,Correct,True,train,True,False,,True,False,


In [13]:
results_df.to_csv("../results/results1.csv")

In [23]:
results_df = pd.read_csv("../results/results1.csv", index_col=0)

In [24]:
import pandas as pd
import json
import re
from vllm import SamplingParams
from pydantic import BaseModel

# Schema for evaluation task 
class MisconceptionMatch(BaseModel):
    is_match: bool
    reasoning: str

json_schema = MisconceptionMatch.model_json_schema()

# Filter to rows where there is an actual misunderstanding
misunderstanding_df = results_df[(results_df['is_misunderstanding'] == True) & results_df['label_is_misunderstanding'] == True].copy()
print(f"Evaluating {len(misunderstanding_df)} rows where is_misunderstanding == True")


# Prompt
def build_match_prompt(row):
    return f"""You are evaluating whether a model's description of a student's mathematical 
misunderstanding matches a human expert's label for that misunderstanding.

EXPERT LABEL: {row['label_which_misunderstanding']}
MODEL DESCRIPTION: {row['which_misunderstanding']}

Does the model's description accurately capture the same misunderstanding as the expert label?
A match does not require identical wording. It should be considered a match if the model's 
description refers to the same underlying conceptual or procedural error as the expert label, 
even if phrased differently.

Respond with a JSON object containing:
- is_match: true if the model's description matches the expert label, false otherwise
- reasoning: one sentence explaining why they match or don't match
"""

# Build conversations
conversations = [
    [{"role": "user", "content": build_match_prompt(row)}]
    for _, row in misunderstanding_df.iterrows()
]

# Sampling params with guided decoding
try:
    from vllm.sampling_params import StructuredOutputsParams
    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=300,
        structured_outputs=StructuredOutputsParams(json=json_schema),
    )
except ImportError:
    from vllm.sampling_params import GuidedDecodingParams
    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=300,
        guided_decoding=GuidedDecodingParams(json=json_schema),
    )

# Run inference
outputs = llm.chat(
    conversations,
    sampling_params,
    chat_template_kwargs={"enable_thinking": False},
)

# Parse results
def parse_match_response(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(text)
        return {
            "is_match": parsed.get("is_match"),
            "match_reasoning": parsed.get("reasoning", ""),
        }
    except json.JSONDecodeError:
        return {"is_match": None, "match_reasoning": None}

parsed = [parse_match_response(o.outputs[0].text) for o in outputs]

# Attach back to the filtered df
misunderstanding_df['is_match'] = [p['is_match'] for p in parsed]
misunderstanding_df['match_reasoning'] = [p['match_reasoning'] for p in parsed]

# Summary
print(f"\nParse failures: {misunderstanding_df['is_match'].isna().sum()}")
print(f"\nMatch rate: {misunderstanding_df['is_match'].mean():.1%}")
print(f"\nSample of mismatches:")
misunderstanding_df

Evaluating 231 rows where is_misunderstanding == True


Rendering conversations:   0%|          | 0/231 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 231/231 [00:31<00:00,  7.44it/s, est. speed input: 1111.68 toks/s, output: 464.25 toks/s]


Parse failures: 0

Match rate: 42.9%

Sample of mismatches:


,QuestionId,Question.Text,MC_Answer,StudentExplanation,FinalCode,Correct,split,label_is_correct,label_is_misunderstanding,label_which_misunderstanding,is_correct,is_misunderstanding,which_misunderstanding,is_match,match_reasoning
0,33472,\( \frac{1}{3}+\frac{2}{5}= \),\( \frac{3}{15} \),3 and 5 go into 15 so the domminater is 15 and...,Denominator-only change,False,train,False,True,Denominator-only change,False,True,Adding numerators and denominators independently,False,The expert label refers specifically to a misu...
6,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),becausee there are 9 total and three are not s...,Incomplete,False,train,False,True,Incomplete,False,True,whole-number bias,True,The model's description of 'whole-number bias'...
7,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),you do what do you do to the bottom to the top...,Additive,True,train,False,True,Additive,False,True,procedural,False,The expert label 'Additive' refers to a specif...
11,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 9 \),I think this because the top will all ways sta...,WNB,False,train,False,True,WNB,False,True,whole-number bias,True,The model's description 'whole-number bias' ac...
14,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),There are nine in total and 6 are shaded. 9-6=3.,Incomplete,False,train,False,True,Incomplete,False,True,whole-number bias,True,The model's description of 'whole-number bias'...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984,31777,A box contains \( 120 \) counters. The counter...,\( 48 \),"Since 120 divided by 5 is 24, and 24 times by ...",Wrong Fraction,False,train,False,True,Wrong Fraction,False,True,fraction-of-whole,True,The model's description 'fraction-of-whole' re...
985,31774,Calculate \( \frac{1}{2} \div 6 \),\( 3 \),I think this because with fractions you divide...,FlipChange,False,train,False,True,FlipChange,False,True,procedural,False,The expert label 'FlipChange' refers to a spec...
986,32833,Calculate \( \frac{2}{3} \times 5 \),\( \frac{2}{15} \),because say the 5 was 1/5 times them together ...,Inversion,False,train,False,True,Inversion,False,True,misapplying fraction multiplication,True,The model's description of 'misapplying fracti...
991,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),because there are 9 triangles all together and...,Incomplete,False,train,False,True,Incomplete,False,True,whole-number bias,True,The model's description of 'whole-number bias'...


In [25]:
results_df = results_df.merge(
    misunderstanding_df[['QuestionId', 'MC_Answer', 'StudentExplanation', 'is_match', 'match_reasoning']],
    on=['QuestionId', 'MC_Answer', 'StudentExplanation'],
    how='left'
)

In [26]:
results_df.head(20)

,QuestionId,Question.Text,MC_Answer,StudentExplanation,FinalCode,Correct,split,label_is_correct,label_is_misunderstanding,label_which_misunderstanding,is_correct,is_misunderstanding,which_misunderstanding,is_match,match_reasoning
0,33472,\( \frac{1}{3}+\frac{2}{5}= \),\( \frac{3}{15} \),3 and 5 go into 15 so the domminater is 15 and...,Denominator-only change,False,train,False,True,Denominator-only change,False,True,Adding numerators and denominators independently,False,The expert label refers specifically to a misu...
1,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),the dots are going up in 4,Correct,True,train,True,False,NaN,True,False,NaN,NaN,NaN
2,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),because it goes up by five and then u add two ...,Neither,True,train,False,False,NaN,False,True,inconsistent pattern reasoning,NaN,NaN
3,31777,A box contains \( 120 \) counters. The counter...,\( 72 \),I think this because 1/5 of 120 is 24 and 3/5 ...,Correct,True,train,True,False,NaN,True,False,NaN,NaN,NaN
4,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 3 \),i was just guessing because i haven't gone ove...,Neither,False,train,False,False,NaN,False,False,NaN,NaN,NaN
5,91695,Dots have been arranged in these patterns: [Im...,\( 22 \),So we addd four in of them 18+4 in total.,Wrong term,False,train,False,True,Wrong term,True,False,NaN,NaN,NaN
6,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),becausee there are 9 total and three are not s...,Incomplete,False,train,False,True,Incomplete,False,True,whole-number bias,True,The model's description of 'whole-number bias'...
7,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),you do what do you do to the bottom to the top...,Additive,True,train,False,True,Additive,False,True,procedural,False,The expert label 'Additive' refers to a specif...
8,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),6/10 is equivalent to 9/15 because 5x2=10 and ...,Correct,True,train,True,False,NaN,False,True,whole-number bias,NaN,NaN
9,32833,Calculate \( \frac{2}{3} \times 5 \),\( 3 \frac{1}{3} \),i think this is because i did 2 1/3 times 5/1 ...,Correct,True,train,True,False,NaN,True,False,NaN,NaN,NaN


In [27]:
results_df.to_csv('../results/results2.csv')

## Classification Report

In [43]:
from sklearn.metrics import classification_report

eval_df = results_df.copy()

eval_df = eval_df.dropna(subset=['is_correct', 'is_misunderstanding'])
eval_df['is_correct'] = eval_df['is_correct'].astype(bool)
eval_df['is_misunderstanding'] = eval_df['is_misunderstanding'].astype(bool)
eval_df['label_is_correct'] = eval_df['label_is_correct'].astype(bool)
eval_df['label_is_misunderstanding'] = eval_df['label_is_misunderstanding'].astype(bool)

# Classify is_correct on all responses
print("Classification Report: is_correct")
print(classification_report(
    eval_df['label_is_correct'],
    eval_df['is_correct'],
    target_names=['Incorrect', 'Correct'],
))

# Only calculate accuracy on incorrect responses
print("Classification Report: is_misunderstanding")
print(classification_report(
    eval_df['label_is_misunderstanding'],
    eval_df['is_misunderstanding'],
    target_names=['Not Misconception', 'Misconception'],
))

# Check how accurate the misunderstanding classification was
print(f'Overall Reasoning Accuracy for Misunderstanding {eval_df['is_match'].mean()}')
print('\n')

for misunderstanding in results_df['label_which_misunderstanding'].dropna().unique():
    print(misunderstanding)
    print(f'N = {len(results_df[results_df['label_which_misunderstanding']==misunderstanding])}')
    print(f'Accuracy = {results_df[results_df['label_which_misunderstanding']==misunderstanding]['is_match'].mean()}')
    print('\n')

Classification Report: is_correct
              precision    recall  f1-score   support

   Incorrect       0.74      0.94      0.82       577
     Correct       0.86      0.54      0.67       425

    accuracy                           0.77      1002
   macro avg       0.80      0.74      0.74      1002
weighted avg       0.79      0.77      0.76      1002

Classification Report: is_misunderstanding
                   precision    recall  f1-score   support

Not Misconception       0.93      0.50      0.65       740
    Misconception       0.39      0.89      0.54       262

         accuracy                           0.60      1002
        macro avg       0.66      0.69      0.59      1002
     weighted avg       0.79      0.60      0.62      1002

Overall Reasoning Accuracy for Misunderstanding 0.4248927038626609


Denominator-only change
N = 18
Accuracy = 0.1111111111111111


Wrong term
N = 16
Accuracy = 1.0


Incomplete
N = 52
Accuracy = 0.8888888888888888


Additive
N = 44
Accura